In [1]:
import sys

import itertools

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from IPython.display import clear_output

In [2]:
sys.path.append("../model-collection/")
from framepool import FramePool, get_optimizer, get_scheduler

sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
model_name = "framepool"
utr_type = "utr3"
device_id = 0 if utr_type == "utr5" else 1
seqsize = 50 if utr_type == "utr5" else 240

In [4]:
PATH_FROM = f"../../predictor/regression_multiple/{utr_type.upper()}_zscores_replicateagg.csv"
df = pd.read_csv(PATH_FROM)

In [5]:
n_splits = 10  # Remaining splits for the 'train' set

df['fold'] = df['fold'].map({'train': -1, 'val': n_splits - 2, 'test': n_splits - 1})

df_unsplit = df.loc[df['fold'] == -1, ['seq', 'fold']].copy()
df_unsplit['fold'] = df_unsplit['seq'].map(
    {seq: split_no
     for seq, split_no in
     zip(df_unsplit['seq'].drop_duplicates(),
         itertools.cycle(range(n_splits - 2)))
     }
)
df.loc[df_unsplit.index, 'fold'] = df_unsplit['fold']

In [6]:
fold_sizes = df['fold'].value_counts()
fold_size = fold_sizes.max()
fold_sizes

fold
0    17058
1    17058
2    17058
3    17058
4    17058
8    17058
5    17052
6    17052
7    17052
9    17052
Name: count, dtype: int64

In [7]:
num_classes = df["cell_type"].unique().shape[0]
num_classes

6

In [8]:
splits = dict(tuple(df.groupby('fold')))
for split_df in splits.values():
    split_df.reset_index(drop=True, inplace=True)
splits[8].head()

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
0,AAAAAAAGGATATAGCAAACCTAACTGATAACTTTGCTAGTTTCAA...,c1,8,6.002917,7.654993,8.144573,14.842668,2.868519,2.466073,0.402446,1.862168,0.216117
1,AAAAAAAGGATATAGCAAACCTAACTGATAACTTTGCTAGTTTCAA...,c13,8,37.271653,34.714706,30.423658,39.155814,2.504808,2.466073,0.038735,0.179234,0.216117
2,AAAAAAAGGATATAGCAAACCTAACTGATAACTTTGCTAGTTTCAA...,c17,8,60.550851,49.969473,28.877094,38.700174,2.256750,2.466073,-0.209323,-0.968564,0.216117
3,AAAAAAAGGATATAGCAAACCTAACTGATAACTTTGCTAGTTTCAA...,c2,8,36.093004,21.819718,25.572823,31.869453,2.461347,2.466073,-0.004726,-0.021866,0.216117
4,AAAAAAAGGATATAGCAAACCTAACTGATAACTTTGCTAGTTTCAA...,c4,8,25.828282,15.219701,18.766588,16.401338,2.337738,2.466073,-0.128335,-0.593825,0.216117


In [9]:
batch_size = 128
train_size = fold_size * (n_splits - 2)
steps_per_epoch = max(1, int(train_size // batch_size))

epochs = None

In [10]:
num_workers = 32

In [11]:
folds_arg = [{
    'train_folds': {j % n_splits for j in range(i, i + n_splits - 2)},
    'val_fold': (i - 2) % n_splits,
    'test_fold': (i - 1) % n_splits
} for i in range(n_splits)]
folds_arg

[{'train_folds': {0, 1, 2, 3, 4, 5, 6, 7}, 'val_fold': 8, 'test_fold': 9},
 {'train_folds': {1, 2, 3, 4, 5, 6, 7, 8}, 'val_fold': 9, 'test_fold': 0},
 {'train_folds': {2, 3, 4, 5, 6, 7, 8, 9}, 'val_fold': 0, 'test_fold': 1},
 {'train_folds': {0, 3, 4, 5, 6, 7, 8, 9}, 'val_fold': 1, 'test_fold': 2},
 {'train_folds': {0, 1, 4, 5, 6, 7, 8, 9}, 'val_fold': 2, 'test_fold': 3},
 {'train_folds': {0, 1, 2, 5, 6, 7, 8, 9}, 'val_fold': 3, 'test_fold': 4},
 {'train_folds': {0, 1, 2, 3, 6, 7, 8, 9}, 'val_fold': 4, 'test_fold': 5},
 {'train_folds': {0, 1, 2, 3, 4, 7, 8, 9}, 'val_fold': 5, 'test_fold': 6},
 {'train_folds': {0, 1, 2, 3, 4, 5, 8, 9}, 'val_fold': 6, 'test_fold': 7},
 {'train_folds': {0, 1, 2, 3, 4, 5, 6, 9}, 'val_fold': 7, 'test_fold': 8}]

In [12]:
checked = {
    "seed": [3],
    "folds": folds_arg,
    "features": [
        ("sequence", "conditions"),
    ],
    "augment_dict": [
        dict(
            extend_left=0,
            extend_right=0,
            shift_left=0,
            shift_right=0,
            revcomp=False,
        ),
    ],
    "epochs": [15],
}

In [13]:
def launch_model(
    seed: int,
    train_ds_kws: dict,
    val_ds_kws: dict,
    model_class,
    model_kws: dict,
    criterion_class,
    criterion_kws: dict,
    optimizer_class,
    optimizer_kws: dict,
    lr_scheduler_class,
    lr_scheduler_kws: dict,
    test_time_validation: bool,
    train_folds: set,
    val_fold: int,
    test_fold: int,
    initialize_weights: bool,
    epochs: int = epochs,
):
    pl.seed_everything(seed)

    # Creating Datasets
    train_set = utrdata.UTRData(
        df=pd.concat([splits[i] for i in train_folds]),
        **train_ds_kws,
    )
    val_set = utrdata.UTRData(
        df=splits[val_fold],
        **val_ds_kws,
    )
    test_set = utrdata.UTRData(
        df=splits[test_fold],
        **val_ds_kws,
    )

    assert train_set.num_channels == val_set.num_channels
    try:
        div_factor = val_ds_kws["augment_kws"]["shift_left"] + \
                     val_ds_kws["augment_kws"]["shift_right"] + 1
    except KeyError:
        div_factor = 1

    # Creating DataLoaders
    dl_train = DataLoader(
        train_set,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True
    )
    dl_val = DataLoader(
        val_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )
    dl_test = DataLoader(
        test_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )

    model = RNARegressor(
        model_class=model_class,
        model_kws=model_kws | dict(
            in_channels=train_set.num_channels
        ),
        criterion_class=criterion_class,
        criterion_kws=criterion_kws,
        optimizer_class=optimizer_class,
        optimizer_kws=optimizer_kws,
        lr_scheduler_class=lr_scheduler_class,
        lr_scheduler_kws=lr_scheduler_kws,
        test_time_validation=test_time_validation,
        initialize_weights=initialize_weights,
    )
    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        dirpath="saved_models",
        save_top_k=1,
        save_last=False,
        monitor="val_pearson_r_0",
        mode="max"
    )
    progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)

    logger = pl.loggers.tensorboard.TensorBoardLogger("tb_logs", name=model.model_name)
    trainer = pl.Trainer(
        callbacks=[checkpoint_callback, progressbar_callback],
        logger=logger,
        accelerator="gpu",
        devices=[device_id],
        # deterministic=True,
        max_epochs=epochs,
        num_sanity_val_steps=0,
        # gradient_clip_val=1e-5,
        # gradient_clip_algorithm="norm",
    )
    trainer.fit(model=model, train_dataloaders=dl_train, val_dataloaders=dl_val)
    best_model = RNARegressor.load_from_checkpoint(checkpoint_callback.best_model_path)

    prediction = trainer.predict(model=best_model, dataloaders=dl_test)
    test_pred, test_real = zip(*prediction)
    test_pred = torch.concat(test_pred).numpy()
    test_real = torch.concat(test_real).numpy()
    test_df = splits[test_fold].copy()
    test_df["real_0"] = test_real[:, 0]
    test_df["pred_0"] = test_pred[:, 0]

    return trainer, test_df

In [ ]:
results = []
for subset in itertools.product(
    *checked.values()
):
    PARAMS = dict(zip(checked.keys(), subset))
    AUGMENT_KEY = any(PARAMS["augment_dict"].values())
    AUGMENT_TEST_TIME = AUGMENT_KEY

    trainer_last, prediction_best_last = launch_model(
        seed=PARAMS["seed"],
        train_ds_kws=dict(
            construct_type=utr_type,
            predict_cols=["mass_center"],
            features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
            augment=AUGMENT_KEY,
            augment_test_time=False,
            augment_kws=PARAMS["augment_dict"],
        ),
        val_ds_kws=dict(
            construct_type=utr_type,
            predict_cols=["mass_center"],
            features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
            augment=False,
            augment_test_time=AUGMENT_TEST_TIME,
            augment_kws=PARAMS["augment_dict"],
        ),
        model_class=FramePool,
        model_kws=dict(
            in_channels=5,
            conv_channels=128,
            out_channels=1,
            n_conv_layers=3,
            kernel_sizes=(7, 7, 7),
            dilations=(1, 1, 1),
            dropouts=(0.0, 0.0, 0.0),
            padding='same',
            skip_connections=True,
            n_fc_layers=1,
            fc_layer_sizes=(64,),
            fc_dropouts=(0.2,),
            only_max_pool=False,
        ),
        criterion_class=nn.MSELoss,
        criterion_kws=dict(),
        **get_optimizer(),
        **get_scheduler(),
        test_time_validation=AUGMENT_TEST_TIME,
        **PARAMS["folds"],
        initialize_weights=False,
        epochs=PARAMS["epochs"],
    )
    results.append((subset, prediction_best_last))

Global seed set to 3
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:612: UserWarning: Checkpoint directory /home/arsen_l/rna-legnet/wp6_clean_repo/parade/benchmark/crossval/saved_models exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type      | Params
----------------------------------------
0 | model  

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=15` reached.
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

Global seed set to 3
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:612: UserWarning: Checkpoint directory /home/arsen_l/rna-legnet/wp6_clean_repo/parade/benchmark/crossval/saved_models exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type      | Params
----------------------------------------
0 | model  

Training: 0it [00:00, ?it/s]

## Validation

In [23]:
full_result = pd.concat([r for _, r in results])
full_result

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std,real_0,pred_0
0,AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...,c1,9,10.930684,7.519106,9.592497,8.605105,2.433101,2.613522,-0.180421,-0.975516,0.184949,2.433101,2.658807
1,AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...,c13,9,13.668966,13.073336,11.705055,11.700622,2.427481,2.613522,-0.186041,-1.005901,0.184949,2.427481,2.549164
2,AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...,c17,9,12.064452,5.648295,6.148830,24.796787,2.897645,2.613522,0.284123,1.536220,0.184949,2.897645,2.734722
3,AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...,c2,9,12.491993,9.447546,17.713293,12.928093,2.591041,2.613522,-0.022481,-0.121553,0.184949,2.591041,2.656542
4,AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...,c4,9,6.787326,2.369534,4.600547,7.061687,2.573348,2.613522,-0.040174,-0.217215,0.184949,2.573349,2.689924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17053,TTTTTTTTTTTTTTTAAAAAAAAAAAAGGTTCAGCAGAAATCAGCG...,c13,8,26.837413,27.927487,23.135730,34.015603,2.574801,2.727882,-0.153081,-0.749808,0.204161,2.574801,2.493034
17054,TTTTTTTTTTTTTTTAAAAAAAAAAAAGGTTCAGCAGAAATCAGCG...,c17,8,38.060705,55.207521,91.092177,145.046166,3.041642,2.727882,0.313760,1.536831,0.204161,3.041642,2.588018
17055,TTTTTTTTTTTTTTTAAAAAAAAAAAAGGTTCAGCAGAAATCAGCG...,c2,8,27.239004,17.142470,10.811631,31.460672,2.536549,2.727882,-0.191333,-0.937170,0.204161,2.536549,2.458018
17056,TTTTTTTTTTTTTTTAAAAAAAAAAAAGGTTCAGCAGAAATCAGCG...,c4,8,13.711310,22.556144,26.920033,38.907618,2.891560,2.727882,0.163679,0.801715,0.204161,2.891561,2.592172


In [ ]:
full_result.to_csv(f"results/{utr_type}_{model_name}_crossval.csv", index=False)

In [24]:
full_result.groupby('cell_type')[['mass_center', 'pred_0']].corr().unstack(-1)[('mass_center', 'pred_0')]

cell_type
c1     0.720497
c13    0.530784
c17    0.631787
c2     0.672237
c4     0.539642
c6     0.646303
Name: (mass_center, pred_0), dtype: float64

In [25]:
full_result.groupby('seq')[['mass_center', 'pred_0']].mean().corr().stack()[('mass_center', 'pred_0')]

0.7147438481470468

In [26]:
full_result.groupby(['cell_type', 'fold'])[['mass_center', 'pred_0']].corr().unstack(-1)[('mass_center', 'pred_0')].unstack(0)

cell_type,c1,c13,c17,c2,c4,c6
fold,,,,,,
0,0.719204,0.517416,0.646232,0.676551,0.553210,0.643321
1,0.717405,0.548265,0.651740,0.684580,0.541656,0.659162
2,0.727738,0.542122,0.619289,0.665852,0.514341,0.647028
3,0.735416,0.559202,0.623140,0.680732,0.536914,0.646655
4,0.733704,0.509288,0.621352,0.671519,0.532971,0.640491
5,0.720602,0.553584,0.637807,0.673178,0.550021,0.646718
6,0.720886,0.575565,0.658220,0.686453,0.557255,0.668151
7,0.738889,0.510475,0.617489,0.684433,0.532542,0.660677
8,0.720747,0.575711,0.652868,0.698156,0.574740,0.675713


In [27]:
full_result.groupby(['seq', 'fold'])[['mass_center', 'pred_0']].mean().groupby(level='fold').corr().unstack(-1)[('mass_center', 'pred_0')]

fold
0    0.719217
1    0.726938
2    0.706040
3    0.717040
4    0.713332
5    0.719853
6    0.731873
7    0.716996
8    0.741900
9    0.721660
Name: (mass_center, pred_0), dtype: float64

---